# The Theta Method: A Simple Benchmark

The Theta method changes the curvature of a series, forecasts the resulting lines and combines them ([Assimakopoulos and Nikolopoulos (2000)](https://doi.org/10.1016/S0169-2070(00)00066-2)). Durbyn.jl offers four variants, set out by [Fiorucci, Pellegrini, Louzada, Petropoulos and Koehler (2016)](https://doi.org/10.1016/j.ijforecast.2016.02.005). This tutorial fits all four on a trending series, then shows why a seasonal series calls for `auto_theta`.

In [ ]:
Pollis.packages("Durbyn", "Plots")

In [ ]:
using Durbyn, Random, Statistics
import Plots

Random.seed!(7)

mase(actual, predicted, train) = mean(abs.(actual .- predicted)) / mean(abs.(diff(train)))

## A Trending Series with No Seasonality

The drift of this series changes halfway through, which gives the dynamic variants something to follow. The last 20 values are held back.

In [ ]:
n = 120
drift = vcat(fill(0.3, 60), fill(0.8, 60))
y = 50 .+ cumsum(drift .+ 0.8 .* randn(n))
train, test = y[1:end-20], y[end-19:end]
Plots.plot(y, label="series", xlabel="period", title="Trend with a change of drift")

## Comparing the Four Variants

STM fixes theta at 2, and OTM estimates theta. DSTM and DOTM do the same while updating the trend line each period. The period is 1, as the series has no cycle.

In [ ]:
for variant in (STM, OTM, DSTM, DOTM)
    model = theta(train, 1; model_type=variant)
    fc = forecast(model; h=length(test))
    println(rpad(string(variant), 6), "alpha ", round(model.alpha, digits=3), "  theta ", round(model.theta, digits=3),
        "  MASE ", round(mase(test, fc.mean, train), digits=3))
end
println("Naive MASE ", round(mase(test, forecast(naive(train); h=length(test)).mean, train), digits=3))

One holdout is a single trial, so a small gap between variants says little. Set the seed to another value and the order may change.

OTM and DOTM can return a very large theta here, with alpha at 1, far from the value of 2 in the standard method. A value like that is the fit moving to an extreme, not a measured property of the series, so read the forecast from the plot and not from the size of theta.

In [ ]:
model = theta(train, 1; model_type=OTM)
fc = forecast(model; h=length(test))
Plots.plot(train, label="train", xlabel="period", title="OTM forecast with 95% interval")
Plots.plot!(length(train)+1:n, test, label="held back")
Plots.plot!(length(train)+1:n, fc.mean, ribbon=(fc.mean .- fc.lower[:, 2], fc.upper[:, 2] .- fc.mean), label="OTM")

## A Seasonal Series: `theta` and `auto_theta`

`theta(y, period; model_type=OTM)` fits the series as given, with no seasonal adjustment. `auto_theta(y, period)` tests for seasonality at that lag and removes the pattern by decomposition when the test finds it. Its `model` keyword keeps a chosen variant, and without it the variant is chosen by in-sample mean squared error.

In [ ]:
m = 12
n2 = 144
t2 = 1:n2
y2 = 200 .+ 0.2 .* t2 .+ 25 .* sin.(2pi .* t2 ./ m) .+ 3 .* randn(n2)
train2, test2 = y2[1:end-24], y2[end-23:end]
as_given = theta(train2, m; model_type=OTM)
adjusted = auto_theta(train2, m; model=OTM)
println("theta:      adjusted = ", as_given.decompose, ", MASE ", round(mase(test2, forecast(as_given; h=24).mean, train2), digits=3))
println("auto_theta: adjusted = ", adjusted.decompose, ", MASE ", round(mase(test2, forecast(adjusted; h=24).mean, train2), digits=3))

In [ ]:
Plots.plot(test2, label="held back", xlabel="period", title="Seasonal series")
Plots.plot!(forecast(as_given; h=24).mean, label="theta, no adjustment")
Plots.plot!(forecast(adjusted; h=24).mean, label="auto_theta", lw=2)

The `decompose` field reports whether the adjustment was made. A strong trend can hide the cycle from the test, as the test reads the autocorrelation of the series before any trend is removed. So when a cycle is known to exist and `decompose` is `false`, check the seasonal shape of the forecast by plot.

## References

Assimakopoulos, V. and Nikolopoulos, K. (2000). The theta model: a decomposition approach to forecasting. *International Journal of Forecasting* 16(4), 521–530. [doi:10.1016/S0169-2070(00)00066-2](https://doi.org/10.1016/S0169-2070(00)00066-2)

Fiorucci, J. A., Pellegrini, T. R., Louzada, F., Petropoulos, F. and Koehler, A. B. (2016). Models for optimising the theta method and their relationship to state space models. *International Journal of Forecasting* 32(4), 1151–1161. [doi:10.1016/j.ijforecast.2016.02.005](https://doi.org/10.1016/j.ijforecast.2016.02.005)